# Myocardial Infarction Treatment Trends
### A "bring your data" analytical workflow using synthetic EHR data

**What we are doing:** Building a real epidemiological workflow from scratch —
cohort identification, treatment classification, outcome extraction,
and trend visualization — using synthetic patient data generated by Synthea.

> **Data:** Synthetic patients generated by
> [Synthea](https://synthetichealth.github.io/synthea/).
> Contains no real patient data.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CWML/bring_data/blob/main/mi_trends.ipynb)

## Setup — download data files
Run this cell first. It downloads the data from GitHub into a local
`data/` folder. Safe to re-run — skips files that are already present.

In [1]:
import os
import urllib.request

os.makedirs("data", exist_ok=True)

# -----------------------------------------------------------
# Data hosted on GitHub — no Drive mounting needed
# -----------------------------------------------------------
base_url = "https://raw.githubusercontent.com/CWML/bring_data/main/data/"

files = [
    "patients.csv",
    "encounters.csv",
    "conditions.csv",
    "medications.csv",
    "procedures.csv"
]

for filename in files:
    destination = os.path.join("data", filename)
    if not os.path.exists(destination):
        print(f"  Downloading {filename}...")
        urllib.request.urlretrieve(base_url + filename, destination)
    else:
        print(f"  {filename} already present, skipping")

print("\nData files ready:")
for f in sorted(os.listdir("data")):
    size_mb = os.path.getsize(os.path.join("data", f)) / 1_000_000
    print(f"  {f:<30} {size_mb:>8.1f} MB")

HTTPError: HTTP Error 404: Not Found

## Load tables

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

base_path = 'data'

patients    = pd.read_csv(f'{base_path}/patients.csv',    low_memory=False)
encounters  = pd.read_csv(f'{base_path}/encounters.csv',  low_memory=False)
conditions  = pd.read_csv(f'{base_path}/conditions.csv',  low_memory=False)
medications = pd.read_csv(f'{base_path}/medications.csv', low_memory=False)
procedures  = pd.read_csv(f'{base_path}/procedures.csv',  low_memory=False)

print('Tables loaded:')
for name, df in [('patients',    patients),
                 ('encounters',  encounters),
                 ('conditions',  conditions),
                 ('medications', medications),
                 ('procedures',  procedures)]:
    print(f'  {name:<15} {df.shape[0]:>10,} rows   {df.shape[1]:>3} cols')

# -- R equivalent -------------------------------------------
# library(tidyverse)
# base_path   <- 'data'
# patients    <- read_csv(file.path(base_path, 'patients.csv'))
# encounters  <- read_csv(file.path(base_path, 'encounters.csv'))
# conditions  <- read_csv(file.path(base_path, 'conditions.csv'))
# medications <- read_csv(file.path(base_path, 'medications.csv'))
# procedures  <- read_csv(file.path(base_path, 'procedures.csv'))

## Step 1 — Build the MI ED cohort

We identify patients with any myocardial infarction diagnosis who presented
to the emergency department, and restrict to encounters between 2015 and 2024.

**SNOMED codes used:**
| Code | Description |
|---|---|
| 22298006 | Myocardial infarction (disorder) |
| 401303003 | Acute ST segment elevation MI (STEMI) |
| 401314000 | Acute non-ST segment elevation MI (NSTEMI) |

In [ ]:
# All three MI SNOMED codes
MI_CODES = {22298006, 401303003, 401314000}

# Patients with any MI diagnosis
mi_pts = (
    conditions[conditions['CODE'].isin(MI_CODES)][['PATIENT']]
    .drop_duplicates()
)

# Emergency encounters only
ed = encounters[encounters['ENCOUNTERCLASS'] == 'emergency'].copy()

# MI patients with an ED encounter 2015–2024
cohort = ed.merge(mi_pts, on='PATIENT')
cohort['encounter_year'] = pd.to_datetime(cohort['START']).dt.year
cohort = cohort[cohort['encounter_year'].between(2015, 2024)].copy()

print(f'MI ED cohort     : {len(cohort):,} encounters')
print(f'Year range       : {cohort.encounter_year.min()} – {cohort.encounter_year.max()}')
print(f'Unique patients  : {cohort["PATIENT"].nunique():,}')
print()
print('Encounters by year:')
print(cohort.groupby('encounter_year').size().to_string())

# -- R equivalent -------------------------------------------
# MI_CODES <- c(22298006, 401303003, 401314000)
#
# mi_pts <- conditions |>
#   filter(CODE %in% MI_CODES) |>
#   distinct(PATIENT)
#
# cohort <- encounters |>
#   filter(ENCOUNTERCLASS == 'emergency') |>
#   semi_join(mi_pts, by = 'PATIENT') |>
#   mutate(encounter_year = year(as.Date(START))) |>
#   filter(between(encounter_year, 2015, 2024))

## Step 2 — Classify treatment

We use **aspirin administration** as our observed treatment signal —
it appears in the medications table and is a core component of real
MI treatment guidelines. We then simulate a second variable representing
adoption of a guideline-recommended add-on therapy over time, to
illustrate how a treatment shift analysis would be structured.

> **Note:** The adoption curve below is **simulated and injected** —
> it does not reflect an observed trend in this synthetic dataset.
> Synthea does not model the full complexity of real MI pharmacotherapy.
> This is explicit synthetic augmentation to illustrate the analytical
> pattern — not a finding.

In [ ]:
# -----------------------------------------------------------
# Observed treatment: aspirin administration
# This IS present in the Synthea medications table
# -----------------------------------------------------------
had_aspirin = (
    medications[
        medications['DESCRIPTION'].str.contains('aspirin', case=False, na=False)
    ][['PATIENT']]
    .drop_duplicates()
    .assign(had_aspirin=True)
)

cohort = cohort.merge(had_aspirin, on='PATIENT', how='left')
cohort['had_aspirin'] = cohort['had_aspirin'].fillna(False)

print('Aspirin in medications table:')
print(medications[medications['DESCRIPTION'].str.contains(
    'aspirin', case=False, na=False)]['DESCRIPTION'].value_counts().to_string())
print()

# -----------------------------------------------------------
# SIMULATED treatment shift
# Represents documented real-world adoption of dual antiplatelet
# therapy (DAPT) — adding a P2Y12 inhibitor (e.g. clopidogrel,
# ticagrelor) to aspirin — per ACC/AHA guideline updates over
# this period. This is injected, not observed.
# -----------------------------------------------------------
np.random.seed(42)
p_dapt = {
    2015: 0.45, 2016: 0.50, 2017: 0.55, 2018: 0.60,
    2019: 0.65, 2020: 0.68, 2021: 0.72, 2022: 0.76,
    2023: 0.80, 2024: 0.84
}

cohort['p_dapt'] = cohort['encounter_year'].map(p_dapt).fillna(0)
cohort['treatment'] = np.where(
    ~cohort['had_aspirin'],
    'no_aspirin',
    np.where(
        np.random.random(len(cohort)) < cohort['p_dapt'],
        'aspirin_plus_p2y12',
        'aspirin_only'
    )
)

print('Treatment classification by year:')
print(cohort.groupby(['encounter_year', 'treatment']).size().unstack(fill_value=0))

# -- R equivalent -------------------------------------------
# had_aspirin <- medications |>
#   filter(str_detect(DESCRIPTION, regex('aspirin', ignore_case=TRUE))) |>
#   distinct(PATIENT) |>
#   mutate(had_aspirin = TRUE)
#
# p_dapt <- c('2015'=0.45,'2016'=0.50,'2017'=0.55,'2018'=0.60,
#             '2019'=0.65,'2020'=0.68,'2021'=0.72,'2022'=0.76,
#             '2023'=0.80,'2024'=0.84)
#
# cohort <- cohort |>
#   left_join(had_aspirin, by = 'PATIENT') |>
#   replace_na(list(had_aspirin = FALSE)) |>
#   mutate(
#     p_dapt    = p_dapt[as.character(encounter_year)],
#     treatment = case_when(
#       !had_aspirin              ~ 'no_aspirin',
#       runif(n()) < p_dapt       ~ 'aspirin_plus_p2y12',
#       TRUE                      ~ 'aspirin_only'
#     )
#   )

## Step 3 — Outcomes

In [ ]:
# 30-day mortality: join death date from patients table
cohort = cohort.merge(
    patients[['Id', 'DEATHDATE']].rename(columns={'Id': 'PATIENT'}),
    on='PATIENT', how='left'
)
cohort['encounter_date'] = pd.to_datetime(cohort['START'])
cohort['death_date']     = pd.to_datetime(cohort['DEATHDATE'])
cohort['days_to_death']  = (cohort['death_date'] - cohort['encounter_date']).dt.days
cohort['died_30d']       = cohort['days_to_death'].between(0, 30)

print('Outcome summary:')
print(f"  Any aspirin      : {cohort['had_aspirin'].sum():,}  "
      f"({cohort['had_aspirin'].mean():.1%})")
print(f"  30-day mortality : {cohort['died_30d'].sum():,}  "
      f"({cohort['died_30d'].mean():.1%})")

# -- R equivalent -------------------------------------------
# cohort <- cohort |>
#   left_join(
#     patients |> select(PATIENT = Id, DEATHDATE),
#     by = 'PATIENT'
#   ) |>
#   mutate(
#     days_to_death = as.numeric(as.Date(DEATHDATE) - as.Date(START)),
#     died_30d      = between(days_to_death, 0, 30)
#   )

## Step 4 — Visualize trends

In [ ]:
# Annual summary
annual = (
    cohort.groupby('encounter_year')
    .agg(
        total          = ('Id', 'count'),
        aspirin        = ('had_aspirin', 'sum'),
        dapt           = ('treatment', lambda x: (x == 'aspirin_plus_p2y12').sum()),
        died_30d       = ('died_30d', 'sum')
    )
    .reset_index()
)
annual['aspirin_rate']   = annual['aspirin'] / annual['total']
annual['dapt_share']     = annual['dapt']    / annual['aspirin'].clip(lower=1)
annual['mortality_rate'] = annual['died_30d'] / annual['total']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    'MI Treatment Trends — Synthetic Cohort (Synthea)\n'
    'DAPT adoption is simulated; outcomes are model-generated',
    fontsize=11, color='#555555'
)

blue   = '#4878CF'
orange = '#E87D35'
red    = '#D65F5F'

# Left panel: treatment mix
axes[0].plot(annual['encounter_year'], annual['aspirin_rate'],
             marker='o', color=blue, label='Any aspirin')
axes[0].plot(annual['encounter_year'], annual['dapt_share'],
             marker='s', color=orange, linestyle='--',
             label='DAPT share (simulated)')
axes[0].set_title('Aspirin Use & DAPT Adoption')
axes[0].set_ylabel('Rate')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_xlabel('Year')
axes[0].legend(fontsize=9)
axes[0].spines[['top', 'right']].set_visible(False)

# Right panel: 30-day mortality
axes[1].plot(annual['encounter_year'], annual['mortality_rate'],
             marker='o', color=red, label='30-day mortality')
axes[1].set_title('30-Day Mortality Rate')
axes[1].set_ylabel('Rate')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_xlabel('Year')
axes[1].legend(fontsize=9)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('mi_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: mi_trends.png')

# -- R equivalent -------------------------------------------
# p1 <- annual |>
#   ggplot(aes(x = encounter_year)) +
#   geom_line(aes(y = aspirin_rate, colour = 'Any aspirin'), size=1) +
#   geom_line(aes(y = dapt_share,   colour = 'DAPT share'),
#             linetype = 'dashed', size = 1) +
#   geom_point(aes(y = aspirin_rate), colour = '#4878CF') +
#   scale_y_continuous(labels = scales::percent) +
#   labs(x='Year', y='Rate', colour=NULL,
#        title='Aspirin Use & DAPT Adoption') +
#   theme_minimal()
#
# p2 <- annual |>
#   ggplot(aes(x = encounter_year, y = mortality_rate)) +
#   geom_line(colour = '#D65F5F', size = 1) + geom_point() +
#   scale_y_continuous(labels = scales::percent) +
#   labs(x='Year', y='Rate', title='30-Day Mortality Rate') +
#   theme_minimal()
#
# p1 + p2 +
#   plot_annotation(
#     title   = 'MI Treatment Trends — Synthetic Cohort',
#     caption = 'DAPT adoption simulated; outcomes model-generated'
#   )

## Your turn

**Discuss:**
1. What would you expect the real 30-day mortality trend to look like
   for MI — and why might it differ from what we see in synthetic data?
2. What variables would you want in a real study that are missing or
   limited here? *(Think: time to treatment, ejection fraction,
   Killip class, insurance status, transfer status.)*
3. If you brought your own data or research question today —
   what condition or treatment would you swap in for MI?

---

## Want to go further? Pick one — or ask the AI

1. Split the cohort into STEMI vs NSTEMI and compare treatment rates
2. Stratify thrombolytic use by age group or sex
3. Add a second treatment variable from the `procedures` table
   *(hint: look for percutaneous coronary intervention)*
4. Calculate time from ED arrival to treatment as a process measure
5. Compare comorbidity burden between treated and untreated patients
6. Add a logistic regression predicting 30-day mortality
7. Map MI encounters to Connecticut county — build a choropleth
8. Swap in a completely different condition using its SNOMED code

---

## Curious about what this looks like with real EHR data at scale?

This workflow mirrors the analytical approach used in studies conducted
using [Epic Cosmos](https://www.epic.com/epic/post/cosmos) —
a research platform aggregating EHR data from over 286 million patients.

> *Trends in Thrombolysis in Stroke (2015–2024)*
> Yale School of Medicine — see link shared by your instructor.